# Check CITE-seq counts

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import warnings 
import numpy as np
import pandas as pd
import seaborn as sns
import muon as mu
import anndata as ad


warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, facecolor='white')

#### load in the adata

In [ ]:
adata_protein = sc.read_h5ad("/home/prisb/projects/henry_hashtag_experiment/exmds/combined_protein.h5ad")
adata_protein.obs.index = adata_protein.obs["dataset"].astype('str') + "_" + adata_protein.obs.index


In [ ]:
adata_rna = sc.read_h5ad("09_adata_sc_mut.h5ad")
adata_rna.obs.index = adata_rna.obs["dataset_name"].astype('str') + "_" + adata_rna.obs.index

In [ ]:
mdata = mu.MuData({"rna":adata_rna, "protein":adata_protein})
mu.pp.intersect_obs(mdata)
mdata

#### trasnfer the gex obs to protein obs 

In [ ]:
mdata['protein'].obs['patient_alias'] = mdata['rna'].obs['patient_alias']
mdata['protein'].obs['timepoint_coarse'] = mdata['rna'].obs['timepoint_coarse']
mdata['protein'].obs['new_celltype'] = mdata['rna'].obs['new_celltype']

In [ ]:
mdata['protein'].obs[['patient_alias','dataset']].value_counts()

#### visualize the isotype count distribution per dataset

In [ ]:
isotypes = mdata['protein'].var_names[mdata['protein'].var_names.str.contains("Ctrl")].tolist()

In [ ]:
sc.pl.violin(mdata['protein'],isotypes, groupby='dataset', rotation=90)

In [ ]:
sc.pl.heatmap(mdata['protein'], var_names=isotypes, groupby='dataset')

#### visualize the distribution of total counts across datasets

In [ ]:
mdata['protein'].obs["total_counts"] = mdata['protein'].X.sum(axis=1)

In [ ]:
mdata['protein'].obs["total_counts"]

In [ ]:
#compute summary staistics for each dataset (Coefficient of variation)

batch_stats = mdata['protein'].obs.groupby("dataset")["total_counts"].agg(["mean", "std", "min", "max", "median"])
batch_stats['CV'] = batch_stats['std'] / batch_stats['mean']
batch_stats


In [ ]:
median_counts = batch_stats["median"]
fold_change = median_counts.max() / median_counts.min()
print(f"Fold change in median UMI counts across batches: {fold_change:.2f}")

In [ ]:
sc.pl.violin(mdata['protein'], keys='total_counts', groupby='dataset', rotation=90)

In [ ]:
sc.pl.violin(mdata['protein'], keys='total_counts', groupby='patient_alias', rotation=90)

In [ ]:
sc.pl.violin(mdata['protein'], keys='total_counts', groupby='timepoint_coarse', rotation=90)

In [ ]:
sc.pl.violin(mdata['protein'], keys='total_counts', groupby='new_celltype', rotation=90)

In [ ]:
sc.pp.pca(mdata['protein'])

In [ ]:
sc.pl.pca_variance_ratio(mdata['protein'])

In [ ]:
sc.pl.pca(mdata['protein'], color=["dataset"])